In [ ]:
using RemoteREPL
@async serve_repl()

# Cold Magnetized Plasma Dispersion Relation

---

## Overview

This notebook derives and visualizes the **dispersion relation for waves in a cold, magnetized electron–ion plasma**.

We work in the **Stix notation**, normalized to the **ion cyclotron frequency** $\Omega_i = eB_0/m_i$:

$$
\bar{\omega} = \frac{\omega}{\Omega_i}, \qquad
\mu = \frac{m_i}{m_e}, \qquad
\alpha = \frac{\omega_{pi}^2}{\Omega_i^2}
$$

The dielectric tensor is
$$
\boldsymbol{\varepsilon} = \begin{pmatrix} S & -iD & 0 \\ iD & S & 0 \\ 0 & 0 & P \end{pmatrix}
$$

with the **Stix parameters**:
$$
S = 1 - \frac{\alpha}{\bar{\omega}^2-1} - \frac{\mu\alpha}{\bar{\omega}^2-\mu^2}, \qquad
D = \frac{\alpha}{\bar{\omega}(\bar{\omega}^2-1)} - \frac{\mu^2\alpha}{\bar{\omega}(\bar{\omega}^2-\mu^2)}, \qquad
P = 1 - \frac{(1+\mu)\alpha}{\bar{\omega}^2}
$$

and
$$
R = S + D, \qquad L = S - D.
$$

---

## 1. Symbolic Setup

We use **Symbolics.jl** to define the Stix parameters as exact rational functions of $\bar{\omega}$, then derive the dispersion relation symbolically for propagation parallel and perpendicular to $\mathbf{B}_0$.

In [ ]:
using Symbolics, Latexify, LinearAlgebra

# Declare symbolic variables
@variables ω̄ α μ θ n²

# ── Stix parameters ──────────────────────────────────────────────────────────
S_sym = 1 - α/(ω̄^2 - 1) - μ*α/(ω̄^2 - μ^2)
D_sym = α/(ω̄*(ω̄^2 - 1)) - μ^2*α/(ω̄*(ω̄^2 - μ^2))
P_sym = 1 - (1 + μ)*α/ω̄^2

R_sym = S_sym + D_sym
L_sym = S_sym - D_sym

println("Stix S = "); display(S_sym)
println("Stix D = "); display(D_sym)
println("Stix P = "); display(P_sym)

In [ ]:
function taylor_expand(expr, x, x0, order)
    terms = []
    
    for n in 0:order
        d = expr
        for _ in 1:n
            d = Symbolics.derivative(d, x)
        end
        
        coeff = Symbolics.substitute(d, Dict(x => x0)) / factorial(n)
        push!(terms, coeff * (x - x0)^n)
    end
    
    return sum(terms)
end

In [ ]:
# S_sym = taylor_expand(S_sym, μ , 0.0, 2)
# D_sym = taylor_expand(D_sym, μ , 0.0, 2)
# P_sym = taylor_expand(P_sym, μ , 0.0, 2)
# R_sym = S_sym + D_sym
# L_sym = S_sym - D_sym

## 2. Dispersion Relation: Appleton–Hartree Form

For a wave propagating at angle $\theta$ to $\mathbf{B}_0$, the cold-plasma dispersion relation (Appleton–Hartree) is

$$
An^4 - Bn^2 + C = 0
$$

where $n^2 = k^2 c^2/\omega^2$ is the **squared refractive index**, and

$$
A = S\sin^2\theta + P\cos^2\theta, \qquad
B = RL\sin^2\theta + PS(1+\cos^2\theta), \qquad
C = PRL.
$$

The two roots give the **ordinary (O) and extraordinary (X)** modes for perpendicular propagation, and the **R and L whistler/Alfvén** modes for parallel propagation.

In [ ]:
# Symbolic Appleton–Hartree coefficients
A_sym = S_sym*sin(θ)^2 + P_sym*cos(θ)^2
B_sym = R_sym*L_sym*sin(θ)^2 + P_sym*S_sym*(1 + cos(θ)^2)
C_sym = P_sym*R_sym*L_sym

println("\nAppleton–Hartree coefficients (symbolic):")
println("A = "); display(A_sym)
println("B = "); display(B_sym)
println("C = "); display(C_sym)

### Parallel propagation ($\theta = 0$)

Setting $\theta = 0$ the dispersion relation factors cleanly:

$$
P = 0 \quad (\text{plasma cut-off}), \qquad n^2 = R, \qquad n^2 = L.
$$

The $n^2 = R$ branch is the **right-hand circularly polarized (R-wave)**, hosting the **electron whistler** at $\bar\omega \ll \mu$ and the **ion cyclotron** resonance at $\bar\omega = 1$.

The $n^2 = L$ branch is the **left-hand circularly polarized (L-wave)**, propagating as the **shear Alfvén / ion cyclotron wave** for $\bar\omega < 1$.

### Perpendicular propagation ($\theta = \pi/2$)

Setting $\theta = \pi/2$ the quartic factors as

$$
n^2 = P \quad (\text{O-mode}), \qquad n^2 = \frac{RL}{S} \quad (\text{X-mode}).
$$

The **O-mode** is independent of $\mathbf{B}_0$; it cuts off at $\omega = \omega_{pe}$.

The **X-mode** has hybrid resonances at $S = 0$ (upper/lower hybrid frequencies).

---

## 3. Numerical Evaluation and Dispersion Plots

We now evaluate $n^2(\bar\omega)$ numerically for physically motivated parameters.

### Physical parameters

| Symbol | Value | Meaning |
|--------|-------|---------|
| $\mu$  | 1836  | Proton–electron mass ratio |
| $\alpha$ | 10 | $\omega_{pi}^2/\Omega_i^2$ — measures plasma density |

With $\alpha = 10$, $\omega_{pi} \approx 3.16\,\Omega_i$, placing us in a **strongly magnetized** regime but with a substantial plasma density.

In [ ]:
using Plots, LaTeXStrings
default(fontfamily="Computer Modern", framestyle=:box, grid=true,
        gridcolor=:lightgrey, gridlinewidth=0.5, minorgrid=false,
        legendfontsize=10, titlefontsize=13, labelfontsize=12)

# ── Physical parameters ────────────────────────────────────────────────────
const μ_val   = 100.0   # proton/electron mass ratio
const α_val   = 10.0     # ω_pi² / Ω_i²

# ── Stix parameters as plain Julia functions ───────────────────────────────
function stix(ωval, αval=α_val, μval=μ_val)
    subs = Dict(
        ω̄ => ωval,
        α => αval,
        μ => μval,
    )

    S = Symbolics.value(substitute(S_sym, subs)) |> Float64
    D = Symbolics.value(substitute(D_sym, subs)) |> Float64
    P = Symbolics.value(substitute(P_sym, subs)) |> Float64
    R = Symbolics.value(substitute(R_sym, subs)) |> Float64
    L = Symbolics.value(substitute(L_sym, subs)) |> Float64

    return S, D, P, R, L
end

## 4. Stix Parameters vs Frequency

Before plotting wave modes, it is instructive to visualize $R$, $L$, $S$, $D$, $P$ as functions of $\bar\omega$.

Key features to note:
- **Resonances** (divergences) occur at $\bar\omega = 1$ (ion cyclotron) and $\bar\omega = \mu$ (electron cyclotron)
- **Cut-offs** (zeros) give the edges of propagation bands
- $P = 0$ at the **plasma frequency** $\bar\omega = \omega_{pe}/\Omega_i = \sqrt{\mu\alpha}$

In [ ]:
ω_ion = range(0.01, 2*μ_val, length=5000)
S_i = [stix(ω)[1] for ω in ω_ion]
D_i = [stix(ω)[2] for ω in ω_ion]
P_i = [stix(ω)[3] for ω in ω_ion]
R_i = [stix(ω)[4] for ω in ω_ion]
L_i = [stix(ω)[5] for ω in ω_ion]

p1 = plot(ω_ion, R_i, label=L"R", lw=2,xscale=:log10, legend=:bottomleft)
plot!(p1, ω_ion, L_i, label=L"L", lw=2)
plot!(p1, ω_ion, S_i, label=L"S", lw=2)
plot!(p1, ω_ion, P_i, label=L"P", lw=2)

vline!(p1, [1.0], color=:grey, ls=:dashdot, lw=1, label=L"\Omega_i")
vline!(p1, [μ_val], color=:grey, ls=:dot, lw=1, label=L"\Omega_e")

ylims!(p1, -20, 20)
xlabel!(p1, L"\bar{\omega} = \omega/\Omega_i")
ylabel!(p1, "Stix parameter")
title!(p1, "Stix Parameters — Ion Frequency Range")
display(p1)

## 5. Parallel Propagation ($\theta = 0$): R- and L-waves

For $\theta = 0$ the two modes are $n^2 = R$ and $n^2 = L$.

### Physical branches:

| Branch | Polarization | Key feature |
|--------|-------------|-------------|
| $n^2 = R$ | Right-hand circular | Resonance at $\bar\omega = 1$ (ion cyclotron); passes through as **whistler** for $1 < \bar\omega \ll \mu$ |
| $n^2 = L$ | Left-hand circular | Resonance at $\bar\omega = \mu$ (electron cyclotron); **ion Alfvén** for $\bar\omega \ll 1$ |

When $n^2 > 0$: **propagating wave.** When $n^2 < 0$: **evanescent (cut-off).**

In [ ]:
# Mask near resonances for clean plots
function mask_near(arr, vals, tol=0.03)
    mask = ones(Bool, length(arr))
    tols = isa(tol, Number) ? fill(tol, length(vals)) : collect(tol)
    length(tols) == length(vals) || throw(ArgumentError("tol must be a scalar or match length(vals)"))
    for (v, tv) in zip(vals, tols)
        mask .&= abs.(arr .- v) .> tv
    end
    return mask
end

# ── Ion range dispersion: 0.01 → 1.8 ─────────────────────────────────────
ω_p = range(0.01, 2*μ_val, length=5000)

R_par = [stix(ω)[4] for ω in ω_p]
L_par = [stix(ω)[5] for ω in ω_p]

# Replace negatives with NaN for log plots
R_pos = replace(x -> x <= 0 ? NaN : x, R_par)
L_pos = replace(x -> x <= 0 ? NaN : x, L_par)

p_par1 = plot(ω_p, R_pos, label=L"n^2 = R",
              lw=2.5, yscale=:log10, xscale=:log10, legend=:bottomleft)
plot!(p_par1, ω_p, L_pos, label=L"n^2 = L",
      lw=2.5)
vline!(p_par1, [1.0], color=:grey, ls=:dash, lw=1.5,
       label=L"\Omega_i")
vline!(p_par1, [μ_val], color=:grey, ls=:dot, lw=1.5,
       label=L"\Omega_e")
xlabel!(p_par1, L"\bar{\omega} = \omega/\Omega_i")
ylabel!(p_par1, L"n^2")
title!(p_par1, L"$\theta=0$")
ylims!(p_par1, 1e-1, 1e4)
display(p_par1)

## 6. Perpendicular Propagation ($\theta = \pi/2$): O- and X-modes

For $\theta = \pi/2$ the two modes decouple:

$$
n^2_{\text{O}} = P, \qquad n^2_{\text{X}} = \frac{RL}{S}.
$$

The **O-mode** (ordinary) has the same cut-off as an unmagnetized plasma: $P = 0 \Rightarrow \omega = \omega_{pe}$.

The **X-mode** (extraordinary) is sensitive to $\mathbf{B}_0$ and has:
- **Resonances** at $S = 0$ → lower hybrid $\omega_{LH}$ and upper hybrid $\omega_{UH}$ frequencies
- **Cut-offs** at $R = 0$ and $L = 0$

In [ ]:
ω_px = range(0.01, 1000.0, length=8000)

n2_O = Float64[]
n2_X = Float64[]
for ω in ω_px
    S, D, P, R, L = stix(ω)
    push!(n2_O, P)
    push!(n2_X, abs(S) > 1e-6 ? R*L/S : NaN)
end

n2_O_pos = replace(x -> x <= 0 ? NaN : x, n2_O)
n2_X_pos = replace(x -> (isnan(x) || x <= 0) ? NaN : x, n2_X)

# Compute plasma frequency (P=0 → ω_pe/Ω_i = sqrt(μα))
ω_pe_norm = sqrt(μ_val * α_val)

p_perp = plot(ω_px, n2_O_pos, label=L"n^2_O = P",
              lw=2.5, yscale=:log10, xscale=:log10, legend=:bottomleft)
plot!(p_perp, ω_px, n2_X_pos,
      label=L"n^2_X = RL/S",
      lw=2.5)
vline!(p_perp, [1.0], color=:grey, ls=:dash, lw=1.5,
       label=L"\Omega_i")
vline!(p_perp, [μ_val], color=:grey, ls=:dot, lw=1.5,
       label=L"\Omega_e")
vline!(p_perp, [sqrt(μ_val)], color=:grey, ls=:dot, lw=1.5,
       label=L"\Omega_{LH}")
xlabel!(p_perp, L"\bar{\omega} = \omega/\Omega_i")
ylabel!(p_perp, L"n^2")
title!(p_perp, L"$\theta=90°$")
ylims!(p_perp, 1e-3, 1e3)
display(p_perp)